# **Linear Regression model - our Baseline**

In [18]:
# Install missing libraries if needed
!pip install yfinance

In [23]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [24]:
# Load tickers from CSV file
tickers_df = pd.read_csv('tickers.csv')
tickers = tickers_df.iloc[:, 0].dropna().unique().tolist()

# Download all tickers in bulk
data = yf.download(tickers, start="2020-01-01", end="2024-12-31")['Close']

# Define MAPE function
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

results = []

# Loop through each stock
for ticker in data.columns:
    try:
        df = pd.DataFrame({'Close': data[ticker]})
        df.dropna(inplace=True)

        if df.empty:
            print(f"Skipping {ticker}: No data after cleaning.")
            continue

        # Feature engineering
        df['Day'] = np.arange(len(df))
        df['Target'] = df['Close'].shift(-1)
        df.dropna(inplace=True)

        X = df[['Day']]
        y = df['Target']

        # Chronological split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Metrics
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)

        results.append([ticker, mse, mae, r2, mape])

    except Exception as e:
        print(f"Error processing {ticker}: {e}")

# Save to CSV
results_df = pd.DataFrame(results, columns=['Symbol', 'MSE', 'MAE', 'R2', 'MAPE'])
results_df.to_csv('stock_linear_regression_results.csv', index=False)
print("Saved results to 'stock_linear_regression_results.csv'")


[**********************51%                       ]  260 of 509 completedERROR:yfinance:Could not get exchangeTimezoneName for ticker '???.TA' reason: 'chart'
[*********************100%***********************]  509 of 509 completed
ERROR:yfinance:
15 Failed downloads:
ERROR:yfinance:['CNZN.TA', 'GLTC.TA', 'UNCR.TA', 'INFR-M.TA', 'AICS-M.TA', 'YAAC.TA', 'INTO.TA', 'HDHA.TA', '???.TA', 'ARAD.TA', 'ECPA-M.TA', 'UNVO.TA', 'LVPR.TA', 'PMCN.TA', 'ENDY.TA']: YFTzMissingError('possibly delisted; no timezone found')


Skipping ???.TA: No data after cleaning.
Skipping AICS-M.TA: No data after cleaning.
Skipping ARAD.TA: No data after cleaning.
Skipping CNZN.TA: No data after cleaning.
Skipping ECPA-M.TA: No data after cleaning.
Skipping ENDY.TA: No data after cleaning.
Skipping GLTC.TA: No data after cleaning.
Skipping HDHA.TA: No data after cleaning.
Skipping INFR-M.TA: No data after cleaning.
Skipping INTO.TA: No data after cleaning.
Skipping LVPR.TA: No data after cleaning.
Skipping PMCN.TA: No data after cleaning.
Skipping UNCR.TA: No data after cleaning.
Skipping UNVO.TA: No data after cleaning.
Skipping YAAC.TA: No data after cleaning.
Saved results to 'stock_linear_regression_results.csv'
